# Python Namespaces and Scope Workshop

This notebook explains how Python stores and finds names.

By the end of the workshop, you should be able to:

- explain what a **namespace** is;
- distinguish **built-in, global, local, and enclosing** namespaces;
- explain the difference between a **namespace** and a **scope**;
- use `dir()`, `vars()`, `globals()`, and `locals()`;
- predict how Python searches for a name using the **LEGB rule**;
- understand variable shadowing;
- use `global` and `nonlocal` correctly.

> **Central idea:** a name may be **local to one function** and **enclosing relative to a nested function**.  
> The name is not necessarily duplicated—the description changes according to where it is being accessed.

## 1. What is a namespace?

A **namespace** is a mapping between names and Python objects.

For example:

```python
student = "Alice"
score = 85
```

Python conceptually stores something similar to:

```text
"student" → "Alice"
"score"   → 85
```

A namespace behaves conceptually like a dictionary:

```python
{
    "student": "Alice",
    "score": 85
}
```

Different parts of a program can have different namespaces. This allows the same spelling—such as `result`—to refer to different objects in different functions.

## 2. What is scope?

A **namespace** tells us **where a name is stored**.

A **scope** tells us **where that name can be accessed**.

Example:

```python
def calculate():
    answer = 42
```

`answer` is stored in the local namespace of `calculate()`. Its scope is normally limited to the body of that function.

```text
Namespace: the container holding answer
Scope:     the region where answer can be used
```

## 3. The four namespaces: LEGB

Python normally searches for a name in this order:

| Order | Namespace | Meaning |
|---:|---|---|
| 1 | **Local** | The current function |
| 2 | **Enclosing** | Outer functions surrounding the current function |
| 3 | **Global** | The current Python file or module |
| 4 | **Built-in** | Names supplied automatically by Python |

The initials form **LEGB**:

```text
Local → Enclosing → Global → Built-in
```

Python stops as soon as it finds the requested name.

# Part A — Built-in Namespace

## 4. What is the built-in namespace?

The built-in namespace contains names Python provides automatically.

Examples include:

- functions: `print`, `len`, `sum`, `min`, `max`, `range`;
- types: `int`, `str`, `list`, `dict`, `set`;
- exceptions: `ValueError`, `TypeError`, `NameError`;
- constants: `True`, `False`, `None`.

You can use these names without importing them first.

In [ ]:
# These names are available automatically from Python's built-in namespace.

numbers = [10, 20, 30]

print("Length:", len(numbers))
print("Total:", sum(numbers))
print("Largest:", max(numbers))
print("Type:", type(numbers))

Length: 3
Total: 60
Largest: 30
Type: <class 'list'>


## 5. Inspecting the built-in namespace

The `builtins` module gives us direct access to Python's built-in namespace.

Useful inspection tools:

- `dir(builtins)` returns the available names;
- `vars(builtins)` returns a dictionary-like mapping of names to objects.

In [ ]:
import builtins

# Collect public built-in names.
builtin_names = sorted(
    name for name in dir(builtins)
    if not name.startswith("__")
)

print(f"Number of public built-in names: {len(builtin_names)}")
print("\nPublic names in the built-in namespace:\n")

# Print the names in readable rows.
columns = 5
for start in range(0, len(builtin_names), columns):
    row = builtin_names[start:start + columns]
    print(" | ".join(f"{name:<18}" for name in row))

Number of public built-in names: 153

Public names in the built-in namespace:

ArithmeticError    | AssertionError     | AttributeError     | BaseException      | BaseExceptionGroup
BlockingIOError    | BrokenPipeError    | BufferError        | BytesWarning       | ChildProcessError 
ConnectionAbortedError | ConnectionError    | ConnectionRefusedError | ConnectionResetError | DeprecationWarning
EOFError           | Ellipsis           | EncodingWarning    | EnvironmentError   | Exception         
ExceptionGroup     | False              | FileExistsError    | FileNotFoundError  | FloatingPointError
FutureWarning      | GeneratorExit      | IOError            | ImportError        | ImportWarning     
IndentationError   | IndexError         | InterruptedError   | IsADirectoryError  | KeyError          
KeyboardInterrupt  | LookupError        | MemoryError        | ModuleNotFoundError | NameError         
None               | NotADirectoryError | NotImplemented     | NotImplementedError | O

In [ ]:
# Separate callable built-ins from other built-in objects.

builtin_functions_and_types = []
other_builtin_objects = []

for name in builtin_names:
    value = getattr(builtins, name)

    if callable(value):
        builtin_functions_and_types.append(name)
    else:
        other_builtin_objects.append(name)

print("Callable built-in names:")
print(builtin_functions_and_types)

print("\nOther built-in objects:")
print(other_builtin_objects)

Callable built-in names:
['ArithmeticError', 'AssertionError', 'AttributeError', 'BaseException', 'BaseExceptionGroup', 'BlockingIOError', 'BrokenPipeError', 'BufferError', 'BytesWarning', 'ChildProcessError', 'ConnectionAbortedError', 'ConnectionError', 'ConnectionRefusedError', 'ConnectionResetError', 'DeprecationWarning', 'EOFError', 'EncodingWarning', 'EnvironmentError', 'Exception', 'ExceptionGroup', 'FileExistsError', 'FileNotFoundError', 'FloatingPointError', 'FutureWarning', 'GeneratorExit', 'IOError', 'ImportError', 'ImportWarning', 'IndentationError', 'IndexError', 'InterruptedError', 'IsADirectoryError', 'KeyError', 'KeyboardInterrupt', 'LookupError', 'MemoryError', 'ModuleNotFoundError', 'NameError', 'NotADirectoryError', 'NotImplementedError', 'OSError', 'OverflowError', 'PendingDeprecationWarning', 'PermissionError', 'ProcessLookupError', 'PythonFinalizationError', 'RecursionError', 'ReferenceError', 'ResourceWarning', 'RuntimeError', 'RuntimeWarning', 'StopAsyncIteration

### Built-in lookup example

In the expression below:

```python
print(len(items))
```

Python searches for `print` and `len`.

If these names are not local, enclosing, or global, Python finds them in the built-in namespace.

In [ ]:
items = ["book", "pen", "laptop"]

# print and len come from the built-in namespace.
print("Number of items:", len(items))

Number of items: 3


## 6. Shadowing a built-in name

A global or local variable can hide a built-in name.

This is called **shadowing**.

Avoid using variable names such as `list`, `str`, `sum`, `max`, or `len`.

In [ ]:
# Save the real built-in len function so we can restore normal behaviour.
real_len = builtins.len

# This global variable shadows Python's built-in len function.
len = 100

print("The global name 'len' now refers to:", len)

try:
    print(len([1, 2, 3]))
except TypeError as error:
    print("Error:", error)

# Remove the global variable so later examples can use len normally.
del len

print("Built-in len works again:", real_len([1, 2, 3]))

The global name 'len' now refers to: 100
Error: 'int' object is not callable
Built-in len works again: 3


# Part B — Global Namespace

## 7. What is the global namespace?

A name defined at the top level of a Python file or notebook is normally global.

The global namespace can contain:

- variables;
- functions;
- classes;
- imported modules;
- imported functions or objects.

In a `.py` file, the global namespace belongs to that module. In a notebook, it belongs to the active notebook kernel.

In [ ]:
# Global namespace example

import math                 # Global name: math

name = "Alice"              # Global name: name
age = 25                    # Global name: age

def greet():                # Global name: greet
    # This function reads the global variable name.
    print("Hello,", name)

greet()

Hello, Alice


## 8. Inspecting the global namespace with `globals()`

`globals()` returns a dictionary representing the current global namespace.

Running the exact statement below in a notebook may display many notebook-internal names:

```python
print(globals())
```

For teaching purposes, the next cell prints only the names created in this example.

In [ ]:
print("Initial Global Namespace — selected names:")

selected_global_names = ["math", "name", "age", "greet"]

for global_name in selected_global_names:
    global_object = globals()[global_name]
    print(f"{global_name:<8} → {global_object!r}")

# The exact full command is:
# print(globals())
#
# Uncomment it if you want to display every global name in the notebook.

Initial Global Namespace — selected names:
math     → <module 'math' (built-in)>
name     → 'Alice'
age      → 25
greet    → <function greet at 0x7f11dd52fd10>


Notice that the function name `greet` is also stored globally.

The name points to a function object:

```text
"greet" → <function greet>
```

Defining a function does not merely create executable code—it also creates a name in the current namespace.

## 9. Modifying the global namespace

Changing a top-level variable updates its global binding.

Importing or assigning another object also adds a name to the global namespace.

In [ ]:
# Example solution for the activity

import math

name = "Alice"
age = 25

def greet():
    print("Hello,", name)

print("Initial Global Namespace:")
for global_name in ["math", "name", "age", "greet"]:
    print(f"{global_name:<8} → {globals()[global_name]!r}")

# Modifying the global variables
name = "Bob"
age = 30

print("\nModified Global Namespace:")
for global_name in ["math", "name", "age", "greet"]:
    print(f"{global_name:<8} → {globals()[global_name]!r}")

# Creating another global name that refers to the math module
print("\nAfter creating imported_module:")
imported_module = math

for global_name in ["math", "name", "age", "greet", "imported_module"]:
    print(f"{global_name:<16} → {globals()[global_name]!r}")

Initial Global Namespace:
math     → <module 'math' (built-in)>
name     → 'Alice'
age      → 25
greet    → <function greet at 0x7f11dd5305d0>

Modified Global Namespace:
math     → <module 'math' (built-in)>
name     → 'Bob'
age      → 30
greet    → <function greet at 0x7f11dd5305d0>

After creating imported_module:
math             → <module 'math' (built-in)>
name             → 'Bob'
age              → 30
greet            → <function greet at 0x7f11dd5305d0>
imported_module  → <module 'math' (built-in)>


## 10. Function definitions are global when written at the top level

The next definition replaces the previous global binding called `greet`.

Its parameter `name` and variable `message`, however, are local to each function call.

In [ ]:
def greet(name):
    # name is local because it is a function parameter.
    message = f"Hello, {name}! How are you today?"

    # message is also local because it is assigned inside the function.
    print(message)

# Each call creates a new local namespace.
greet("Alice")
greet("Bob")

print("\nThe name 'greet' remains in the global namespace:")
print(globals()["greet"])

Hello, Alice! How are you today?
Hello, Bob! How are you today?

The name 'greet' remains in the global namespace:
<function greet at 0x7f11dd530cb0>


# Part C — Local Namespace

## 11. What is a local namespace?

Every function call creates a local namespace.

The local namespace normally contains:

- parameters;
- variables assigned inside the function;
- locally defined nested functions;
- locally imported names.

The built-in function `locals()` returns a dictionary representing the current local namespace.

In [ ]:
def demonstrate_locals(student, score):
    # student and score are local parameters.
    status = "Pass" if score >= 50 else "Fail"

    # status is a local variable.
    print("Local namespace inside demonstrate_locals():")
    print(locals())

demonstrate_locals("Alice", 85)

Local namespace inside demonstrate_locals():
{'student': 'Alice', 'score': 85, 'status': 'Pass'}


## 12. Global and local variables can have the same spelling

The following program has two separate bindings named `result`:

- one in the global namespace;
- one in the local namespace of `calculate()`.

The local name is found first while the function is running.

In [ ]:
result = "GLOBAL result"

def calculate():
    # This assignment creates a new LOCAL variable.
    result = "LOCAL result"

    print("Inside calculate():", result)
    print("Local namespace:", locals())

calculate()

# The global result was not changed.
print("Outside calculate():", result)

Inside calculate(): LOCAL result
Local namespace: {'result': 'LOCAL result'}
Outside calculate(): GLOBAL result


The output demonstrates **shadowing**:

```text
Inside the function  → local result
Outside the function → global result
```

Python does not confuse the two because they are stored in different namespaces.

In [ ]:
course = "Python"  # Global variable

def show_course():
    course = "Advanced Python"  # Local variable shadows the global variable

    print("Inside function:", course)
    print("Inside locals():", locals())
    print("Global value using globals():", globals()["course"])

show_course()

print("Outside function:", course)

Inside function: Advanced Python
Inside locals(): {'course': 'Advanced Python'}
Global value using globals(): Python
Outside function: Python


## 13. Local scope and accessibility

A local variable cannot normally be accessed after the function finishes.

In [ ]:
def create_message():
    local_message = "I exist only inside create_message()."
    print(local_message)

create_message()

try:
    # This line is outside the variable's scope.
    print(local_message)
except NameError as error:
    print("Outside the function:", error)

I exist only inside create_message().
Outside the function: name 'local_message' is not defined


# Part D — Enclosing Namespace

## 14. What is an enclosing namespace?

An enclosing namespace exists when a function is defined inside another function.

Terminology:

- the outer function is the **enclosing function**;
- the function defined inside it is a **nested function** or **inner function**;
- the outer function's local namespace is an **enclosing namespace** from the inner function's perspective.

In [ ]:
def outer_function():
    # outer_value is LOCAL to outer_function().
    outer_value = "outer"

    def inner_function():
        # inner_value is LOCAL to inner_function().
        inner_value = "inner"

        print("Inside inner_function():")
        print("outer_value:", outer_value)  # Found in the enclosing namespace
        print("inner_value:", inner_value)  # Found in the local namespace
        print("inner_function locals():", locals())

    print("outer_function locals() before calling inner_function():")
    print(locals())

    inner_function()

outer_function()

outer_function locals() before calling inner_function():
{'inner_function': <function outer_function.<locals>.inner_function at 0x7f11dceb1f50>, 'outer_value': 'outer'}
Inside inner_function():
outer_value: outer
inner_value: inner
inner_function locals(): {'inner_value': 'inner', 'outer_value': 'outer'}


### Why can `outer_value` be called both local and enclosing?

There is only one binding:

```text
outer_function local namespace
└── outer_value → "outer"
```

Its description depends on the point of view:

| Point of view | Role of `outer_value` |
|---|---|
| Code running in `outer_function()` | Local |
| Code running in `inner_function()` | Enclosing |

Therefore:

> `outer_value` is **local to the outer function** and **enclosing relative to the inner function**.

It is not automatically two different variables.

## 15. Accessing an enclosing variable

The nested function can read a name from its nearest outer function.

In [ ]:
def outer_function():
    enclosing_value = "Enclosing Value"

    def nested_function():
        nested_value = "Nested Value"

        # enclosing_value is not local to nested_function().
        # Python finds it in outer_function()'s local namespace.
        print("From enclosing namespace:", enclosing_value)

        # nested_value is local to nested_function().
        print("From local namespace:", nested_value)

    nested_function()

outer_function()

From enclosing namespace: Enclosing Value
From local namespace: Nested Value


The lookup from inside `nested_function()` is:

```text
Search for enclosing_value:
1. Local nested_function()     → not found
2. Enclosing outer_function()  → found
```

## 16. Multiple levels of enclosing functions

Python may search through more than one enclosing function.

In [ ]:
def outer_function():
    enclosing_value = "Enclosing Value"

    def nested_function():
        nested_value = "Nested Value"

        def second_nested():
            # enclosing_value is found two function levels outward.
            print("From outer_function():", enclosing_value)

            # nested_value is found one function level outward.
            print("From nested_function():", nested_value)

            # Show the local namespace of second_nested().
            print("second_nested locals():", locals())

        second_nested()

    nested_function()

outer_function()

From outer_function(): Enclosing Value
From nested_function(): Nested Value
second_nested locals(): {'enclosing_value': 'Enclosing Value', 'nested_value': 'Nested Value'}


From inside `second_nested()`, Python searches outward:

```text
Local second_nested()
→ Enclosing nested_function()
→ Enclosing outer_function()
→ Global
→ Built-in
```

## 17. A nested local variable can shadow an enclosing variable

Assigning `var` inside `nested_function()` creates a new local binding unless `nonlocal` is used.

In [ ]:
def enclosing_function():
    var = "value"  # Local to enclosing_function()

    def nested_function():
        # This is a NEW local variable.
        # It does not modify the enclosing var.
        var = "new_value"
        print("Inside nested_function():", var)

    nested_function()

    # The enclosing variable remains unchanged.
    print("Inside enclosing_function():", var)

enclosing_function()

Inside nested_function(): new_value
Inside enclosing_function(): value


The program has two different bindings:

```text
enclosing_function local namespace
└── var → "value"

nested_function local namespace
└── var → "new_value"
```

## 18. Modifying an enclosing variable with `nonlocal`

Use `nonlocal` when a nested function must modify a binding in the nearest enclosing function.

In [ ]:
def enclosing_function():
    var = "value"

    def nested_function():
        nonlocal var

        # Because of nonlocal, this updates the enclosing binding.
        var = "new_value"
        print("Inside nested_function():", var)

    nested_function()
    print("Inside enclosing_function():", var)

enclosing_function()

Inside nested_function(): new_value
Inside enclosing_function(): new_value


`nonlocal var` means:

> Do not create a new local `var`. Use the nearest `var` stored in an enclosing function namespace.

# Part E — Global Assignment and the `global` Keyword

## 19. Reading a global variable does not require `global`

A function can normally read a global name directly.

In [ ]:
global_var = 10

def read_global():
    # No assignment occurs, so Python can read the global binding.
    print("Read from inside the function:", global_var)

read_global()

Read from inside the function: 10


## 20. Assignment without `global` creates a local variable

In the next example, `global_var = 20` creates a local variable inside `some_function()`.

It does not modify the global variable.

In [ ]:
global_var = 10

def some_function():
    # This is a LOCAL variable because it is assigned inside the function.
    global_var = 20

    print("Inside some_function():", global_var)
    print("Local namespace:", locals())

some_function()

# The global value remains unchanged.
print("Outside some_function():", global_var)

Inside some_function(): 20
Local namespace: {'global_var': 20}
Outside some_function(): 10


## 21. Assignment with `global` modifies the global binding

Use the `global` keyword when a function must assign to a global name.

In [ ]:
global_var = 10

def some_function():
    global global_var

    # This now updates the binding in the global namespace.
    global_var = 20

some_function()

print(global_var)

20


`global global_var` means:

> When this function assigns to `global_var`, use the name in the module's global namespace rather than creating a local variable.

# Part F — Complete LEGB Example

## 22. One program containing all four namespaces

In [ ]:
school = "Data Academy"  # GLOBAL

def outer():
    customer = "Alice"  # LOCAL to outer; ENCLOSING for inner

    def inner(items):
        # items and message are LOCAL to inner.
        message = f"{customer} has {len(items)} items."

        # customer → enclosing
        # school   → global
        # len      → built-in
        # print    → built-in
        # message  → local
        print(message)
        print("School:", school)

    inner(["book", "pen", "laptop"])

outer()

Alice has 3 items.
School: Data Academy


### Classification from inside `inner()`

| Name | Where Python finds it | Reason |
|---|---|---|
| `items` | Local | Parameter of `inner()` |
| `message` | Local | Assigned inside `inner()` |
| `customer` | Enclosing | Defined in `outer()` |
| `school` | Global | Defined at the notebook's top level |
| `len` | Built-in | Supplied by Python |
| `print` | Built-in | Supplied by Python |

Lookup examples:

```text
customer:
Local inner() → Enclosing outer() → FOUND

school:
Local inner() → Enclosing outer() → Global → FOUND

len:
Local inner() → Enclosing outer() → Global → Built-in → FOUND
```

## 23. Inspecting local, enclosing, and global values together

In [ ]:
workshop = "Python Namespaces"  # Global

def teaching_session():
    teacher = "Dr. Lee"  # Local to teaching_session()

    def student_activity(student):
        task = "Classify the names"  # Local to student_activity()

        print("student_activity locals():")
        print(locals())

        print("\nValues obtained from different namespaces:")
        print("Local student:", student)
        print("Local task:", task)
        print("Enclosing teacher:", teacher)
        print("Global workshop:", workshop)
        print("Built-in length:", len(task))

    student_activity("Alice")

teaching_session()

student_activity locals():
{'student': 'Alice', 'task': 'Classify the names', 'teacher': 'Dr. Lee'}

Values obtained from different namespaces:
Local student: Alice
Local task: Classify the names
Enclosing teacher: Dr. Lee
Global workshop: Python Namespaces
Built-in length: 18


# Part G — `locals()` and `globals()` Together

## 24. Comparing the two dictionaries

- `locals()` represents the current local namespace;
- `globals()` represents the module-level global namespace.

At the top level of a notebook, `locals()` and `globals()` often refer to the same active namespace. Inside a function, they are different.

In [ ]:
topic = "Namespaces"  # Global

def compare_namespaces(participant):
    activity = "LEGB exercise"  # Local

    print("Selected local names:")
    for key, value in locals().items():
        print(f"  {key:<12} → {value!r}")

    print("\nSelected global names:")
    for key in ["topic", "compare_namespaces"]:
        print(f"  {key:<20} → {globals()[key]!r}")

compare_namespaces("Alice")

Selected local names:
  participant  → 'Alice'
  activity     → 'LEGB exercise'

Selected global names:
  topic                → 'Namespaces'
  compare_namespaces   → <function compare_namespaces at 0x7f11dceb3cb0>


> **Advanced note:** use `locals()` mainly for inspection. Writing into the dictionary returned by `locals()` inside a function is not a reliable way to change actual local variables.

# Part H — Common Misunderstandings

## 25. “The same name appears in two namespace categories”

Consider:

```python
def outer():
    customer = "Alice"

    def inner():
        print(customer)
```

There is one `customer` binding stored in the local namespace of `outer()`.

From `inner()`:

- it is not local;
- it is found in the surrounding function;
- therefore its role is enclosing.

A clear description is:

```text
customer
Defined in: outer() local namespace
Used from: inner()
Role at that use: enclosing
```

This differs from the following program:

```python
def outer():
    customer = "Alice"

    def inner():
        customer = "Bob"
```

That program creates two separate local bindings with the same spelling.

In [ ]:
def one_binding_example():
    customer = "Alice"

    def inner():
        # Reads the one customer binding from the enclosing function.
        print("One binding:", customer)

    inner()

one_binding_example()


def two_bindings_example():
    customer = "Alice"

    def inner():
        # Creates a different local customer binding.
        customer = "Bob"
        print("Inner local binding:", customer)

    inner()
    print("Outer local binding:", customer)

two_bindings_example()

One binding: Alice
Inner local binding: Bob
Outer local binding: Alice


# Part I — Practice Activities

## Activity 1: Classify every name

For each name, identify whether it is local, enclosing, global, or built-in **from inside `display()`**.

```python
country = "Australia"

def prepare():
    city = "Brisbane"

    def display(person):
        message = f"{person} lives in {city}, {country}"
        print(message)
        print(len(message))

    display("Alice")
```

Classify:

- `person`
- `message`
- `city`
- `country`
- `print`
- `len`

### Activity 1 solution

| Name | Classification inside `display()` |
|---|---|
| `person` | Local |
| `message` | Local |
| `city` | Enclosing |
| `country` | Global |
| `print` | Built-in |
| `len` | Built-in |

In [ ]:
country = "Australia"

def prepare():
    city = "Brisbane"

    def display(person):
        message = f"{person} lives in {city}, {country}"

        print(message)
        print("Message length:", len(message))

    display("Alice")

prepare()

Alice lives in Brisbane, Australia
Message length: 34


## Activity 2: Predict the output

```python
result = 5

def outer():
    result = 10

    def inner():
        result = 15
        print(result)

    inner()
    print(result)

outer()
print(result)
```

Before running the next cell, predict all three lines.

In [ ]:
result = 5

def outer():
    result = 10

    def inner():
        result = 15
        print("inner:", result)

    inner()
    print("outer:", result)

outer()
print("global:", result)

inner: 15
outer: 10
global: 5


### Activity 2 explanation

There are three separate bindings:

```text
inner local result  → 15
outer local result  → 10
global result       → 5
```

Each assignment occurs in a different namespace.

## Activity 3: Update an enclosing value

Complete the function by using `nonlocal` so the final score becomes `20`.

In [ ]:
def score_tracker():
    score = 10

    def add_points():
        nonlocal score
        score = score + 10

    add_points()
    print("Final score:", score)

score_tracker()

Final score: 20


## Activity 4: Update a global value

Complete the function by using `global` so `status` becomes `"open"`.

In [ ]:
status = "closed"

def open_shop():
    global status
    status = "open"

open_shop()
print("Shop status:", status)

Shop status: open


# Part J — Final Summary

## 26. Namespace cheat sheet

| Namespace | Where names come from | Example |
|---|---|---|
| Built-in | Supplied by Python | `print`, `len`, `int` |
| Global | Defined at the module/notebook top level | `school = "UQ"` |
| Local | Parameters and assignments in the current function | `message = "Hello"` |
| Enclosing | Local names from surrounding outer functions | `customer` used by a nested function |

## 27. Keyword cheat sheet

| Tool | Purpose |
|---|---|
| `dir(builtins)` | List built-in names |
| `vars(builtins)` | Inspect built-in name-to-object mappings |
| `globals()` | Inspect the global namespace |
| `locals()` | Inspect the current local namespace |
| `global name` | Assign to a global binding from inside a function |
| `nonlocal name` | Assign to a binding in an enclosing function |

## 28. Final rule

When Python encounters a name, it searches:

```text
Local → Enclosing → Global → Built-in
```

It stops at the first match.

The classification is always relative to the location where the name is being used.